# Modern Quadrant Classification: CRISPR vs RNA-seq

Modern-design quadrant classification scatter plot:
- Clear visual hierarchy
- Elegant color scheme
- Smart label layout
- Professional chart style

In [ ]:
# Load modern visualization packages
library(ggplot2)
library(dplyr)
library(readr)
library(ggrepel)
library(scales)

# Set modern theme
theme_set(theme_minimal(base_size = 14))

cat("✓ Modern visualization environment ready\n")

In [ ]:
# === Parameters and Data Loading ===
INPUT_FILE <- "/path/to/CRISPR/results/crispr_rnaseq_merged.csv"
RESULTS_DIR <- "/path/to/CRISPR/results"

# Threshold settings
Y_THRESHOLD <- 2.0
X_THRESHOLD <- 1.0
PADJ_THRESHOLD <- 0.05

# Highlighted genes
HIGHLIGHT_GENES <- c("ANPEP", "WDR13")

# Modern color scheme - using colors common in scientific journals
modern_colors <- c(
  "Depleted" = "#2E86AB",              # Dark blue
  "Slightly Enriched" = "#F18F01",     # Warm orange
  "Moderately Enriched" = "#E63946",  # Vibrant red
  "Highly Enriched" = "#A4161A"       # Deep red
)

# Highlighted gene colors
highlight_colors <- c(
  "ANPEP" = "#FF6B35",  # Bright orange
  "WDR13" = "#004E89"   # Dark blue
)

cat("✓ Modern color scheme set\n")

In [ ]:
# === Data Preprocessing ===
df <- read_csv(INPUT_FILE, show_col_types = FALSE) %>%
  mutate(
    crispr_score = as.numeric(crispr_score),
    rna_log2fc = as.numeric(rna_log2fc),
    rna_log2fc_abs = abs(rna_log2fc),
    rna_sig = if_else("rna_padj" %in% names(.), rna_padj < PADJ_THRESHOLD, TRUE),
    top_right = (crispr_score >= Y_THRESHOLD) & (rna_log2fc_abs >= X_THRESHOLD) & rna_sig,
    is_highlight = toupper(gene_symbol) %in% toupper(HIGHLIGHT_GENES),
    # Create point size layers
    point_size = case_when(
      is_highlight ~ "Highlight",
      top_right ~ "Important", 
      TRUE ~ "Normal"
    ),
    # Create transparency layers
    point_alpha = case_when(
      is_highlight ~ 1.0,
      top_right ~ 0.8,
      TRUE ~ 0.6
    )
  ) %>%
  filter(!is.na(crispr_score), !is.na(rna_log2fc_abs))

cat(sprintf("Data overview: %d genes, %d top-right, %d highlighted\n", 
            nrow(df), sum(df$top_right), sum(df$is_highlight)))

# Display highlighted gene information
highlight_info <- df %>% 
  filter(is_highlight) %>%
  select(gene_symbol, crispr_score, rna_log2fc, rna_log2fc_abs, crispr_category)

print(highlight_info)

In [ ]:
# === Create Modern Scatter Plot ===
p <- ggplot(df, aes(x = rna_log2fc_abs, y = crispr_score)) +
  
  # Background grid and quadrant area
  geom_rect(aes(xmin = X_THRESHOLD, xmax = Inf, ymin = Y_THRESHOLD, ymax = Inf),
            fill = "#FFF3E0", alpha = 0.3, inherit.aes = FALSE) +
  
  # Threshold lines - modern style
  geom_vline(xintercept = X_THRESHOLD, color = "#666666", 
             linetype = "dashed", size = 0.8, alpha = 0.8) +
  geom_hline(yintercept = Y_THRESHOLD, color = "#666666", 
             linetype = "dashed", size = 0.8, alpha = 0.8) +
  
  # Main scatter points - layered rendering
  geom_point(data = df %>% filter(!is_highlight, !top_right),
             aes(fill = crispr_category, alpha = point_alpha),
             shape = 21, size = 4, stroke = 0.3, color = "white") +
  
  # Top-right region points
  geom_point(data = df %>% filter(top_right, !is_highlight),
             aes(fill = crispr_category),
             shape = 21, size = 5, stroke = 0.8, color = "#333333", alpha = 0.9) +
  
  # Highlighted genes - special treatment
  geom_point(data = df %>% filter(is_highlight),
             aes(fill = crispr_category),
             shape = 21, size = 7, stroke = 2.5, color = "#000000", alpha = 1) +
  
  # Color mapping
  scale_fill_manual(values = modern_colors, name = "CRISPR Category") +
  
  # Axes
  scale_x_continuous(expand = expansion(mult = c(0.02, 0.05))) +
  scale_y_continuous(expand = expansion(mult = c(0.02, 0.05))) +
  
  # Modern theme
  theme_minimal(base_size = 14) +
  theme(
    # Panel settings
    panel.grid.major = element_line(color = "#E5E5E5", size = 0.5),
    panel.grid.minor = element_blank(),
    panel.background = element_rect(fill = "#FAFAFA", color = NA),
    plot.background = element_rect(fill = "white", color = NA),
    
    # Title and labels
    plot.title = element_text(size = 18, face = "bold", color = "#2C3E50", 
                              margin = margin(b = 20)),
    plot.subtitle = element_text(size = 12, color = "#7F8C8D", 
                                 margin = margin(b = 20)),
    axis.title = element_text(size = 14, face = "bold", color = "#34495E"),
    axis.text = element_text(size = 12, color = "#2C3E50"),
    
    # Legend
    legend.position = "right",
    legend.title = element_text(size = 12, face = "bold", color = "#2C3E50"),
    legend.text = element_text(size = 11, color = "#2C3E50"),
    legend.background = element_rect(fill = "white", color = "#E5E5E5", size = 0.5),
    legend.margin = margin(10, 10, 10, 10),
    
    # Margins
    plot.margin = margin(20, 20, 20, 20)
  ) +
  
  # Labels
  labs(
    title = "CRISPR Screen vs RNA-seq Analysis",
    subtitle = "Quadrant classification of gene perturbation effects",
    x = "|RNA-seq log\u2082FC|",
    y = "CRISPR Score (log\u2082FC)",
    caption = "Dashed lines indicate significance thresholds"
  )

print(p)

In [ ]:
# === Smart Label System ===

# Highlighted gene labels - personalized colors
if (sum(df$is_highlight) > 0) {
  highlight_data <- df %>% filter(is_highlight)
  
  for (i in 1:nrow(highlight_data)) {
    gene_info <- highlight_data[i, ]
    gene_name <- gene_info$gene_symbol
    label_color <- highlight_colors[[gene_name]]
    
    p <- p + 
      geom_label_repel(
        data = gene_info,
        aes(label = gene_symbol),
        color = "white",
        fill = label_color,
        fontface = "bold",
        size = 5,
        label.size = 0,
        label.r = unit(0.3, "lines"),
        box.padding = 0.5,
        point.padding = 0.8,
        segment.color = label_color,
        segment.size = 1.2,
        segment.alpha = 0.8,
        min.segment.length = 0.2,
        max.overlaps = 100,
        seed = 42
      )
  }
}

# Top-right gene labels - concise style
top_genes <- df %>% 
  filter(top_right, !is_highlight) %>%
  arrange(desc(crispr_score)) %>%
  slice_head(n = 8)  # Only show the top 8 most important

if (nrow(top_genes) > 0) {
  p <- p + 
    geom_text_repel(
      data = top_genes,
      aes(label = gene_symbol),
      color = "#C0392B",
      fontface = "bold",
      size = 4,
      box.padding = 0.4,
      point.padding = 0.3,
      segment.color = "#E74C3C",
      segment.size = 0.8,
      segment.alpha = 0.7,
      max.overlaps = 15,
      min.segment.length = 0.1,
      seed = 42
    )
}

# Quadrant annotations - modern design
max_x <- max(df$rna_log2fc_abs, na.rm = TRUE)
max_y <- max(df$crispr_score, na.rm = TRUE)

p <- p + 
  # Top-right region annotation
  annotate("text", 
           x = max_x * 0.85, y = max_y * 0.95,
           label = "High Impact\nGenes", 
           color = "#E67E22", fontface = "bold", size = 5,
           hjust = 1, vjust = 1) +
  
  # Threshold annotations
  annotate("text", 
           x = X_THRESHOLD + 0.1, y = max_y * 0.05,
           label = paste("Expression threshold\n|log\u2082FC| \u2265", X_THRESHOLD), 
           color = "#7F8C8D", fontface = "italic", size = 3.5,
           hjust = 0, vjust = 0) +
  
  annotate("text", 
           x = max_x * 0.02, y = Y_THRESHOLD + 0.5,
           label = paste("CRISPR threshold\nScore \u2265", Y_THRESHOLD), 
           color = "#7F8C8D", fontface = "italic", size = 3.5,
           hjust = 0, vjust = 0)

print(p)

In [ ]:
# === Save High-Quality Images ===
if (!dir.exists(RESULTS_DIR)) {
  dir.create(RESULTS_DIR, recursive = TRUE)
}

# Save modern version
ggsave(
  filename = file.path(RESULTS_DIR, "Quadrant_Modern_scatter.pdf"),
  plot = p,
  device = cairo_pdf,
  width = 12, height = 8, units = "in",
  dpi = 300
)

ggsave(
  filename = file.path(RESULTS_DIR, "Quadrant_Modern_scatter.png"),
  plot = p,
  width = 12, height = 8, units = "in",
  dpi = 300
)

cat("✓ Modern charts saved\n")

# Data summary
summary_stats <- df %>%
  summarise(
    total_genes = n(),
    top_right_genes = sum(top_right),
    highlight_genes = sum(is_highlight),
    depleted = sum(crispr_category == "Depleted", na.rm = TRUE),
    enriched = sum(crispr_category %in% c("Slightly Enriched", "Moderately Enriched", "Highly Enriched"), na.rm = TRUE)
  )

cat("\n=== Data Summary ===\n")
cat(sprintf("Total genes: %d\n", summary_stats$total_genes))
cat(sprintf("High-impact genes (top-right): %d\n", summary_stats$top_right_genes))
cat(sprintf("Specially highlighted genes: %d\n", summary_stats$highlight_genes))
cat(sprintf("Depleted genes: %d\n", summary_stats$depleted))
cat(sprintf("Enriched genes: %d\n", summary_stats$enriched))

# Display highlighted gene details
cat("\n=== Highlighted Gene Details ===\n")
highlight_details <- df %>% 
  filter(is_highlight) %>%
  select(gene_symbol, crispr_score, rna_log2fc, rna_log2fc_abs, crispr_category) %>%
  arrange(desc(crispr_score))

print(highlight_details)